<a href="https://colab.research.google.com/github/XTMay/ML_DL/blob/main/LSTM/wikitext_lstm_pytorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 使用PyTorch和WikiText-2数据集训练LSTM文本预测模型

## 学习目标

通过本教程，您将学会：

1. **理解LSTM的基本概念**：长短期记忆网络(LSTM)是如何解决传统RNN的梯度消失问题
2. **掌握文本预处理技术**：如何将原始文本转换为神经网络可以处理的数字序列
3. **实现LSTM模型架构**：使用PyTorch构建完整的LSTM语言模型
4. **训练和评估模型**：学习如何训练语言模型并评估其性能
5. **文本生成技术**：利用训练好的模型生成新的文本内容
6. **高级优化策略**：早停、学习率调度、梯度裁剪等技术

## LSTM简介

**长短期记忆网络(LSTM)**是一种特殊的循环神经网络(RNN)，专门设计来解决长序列的梯度消失问题。LSTM通过引入三个门控机制来控制信息的流动：

- **遗忘门(Forget Gate)**：决定从细胞状态中丢弃什么信息
- **输入门(Input Gate)**：决定什么新信息被存储在细胞状态中
- **输出门(Output Gate)**：决定输出什么部分的细胞状态

这些门控机制使LSTM能够学习何时记住、遗忘或输出信息，从而在处理长序列时保持稳定的梯度流。

---

## 1. 环境设置和依赖库导入

首先安装和导入所需的库。我们将使用PyTorch作为深度学习框架，torchtext用于文本处理。

In [ ]:
# 安装必要的依赖包
!pip install torch torchtext matplotlib seaborn pandas numpy tqdm scikit-learn
!pip install portalocker  # torchtext的依赖

In [ ]:
# 导入必要的库
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torch.nn.functional as F

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter, defaultdict
from tqdm import tqdm
import math
import random
import os
import pickle
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings('ignore')

# 设置中文字体支持
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans', 'Arial Unicode MS', 'sans-serif']
plt.rcParams['axes.unicode_minus'] = False

# 设置随机种子以确保结果可重现
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

# 检查GPU可用性
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'使用设备: {device}')

if torch.cuda.is_available():
    print(f'GPU名称: {torch.cuda.get_device_name(0)}')
    print(f'GPU内存: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x105da24d0>>
Traceback (most recent call last):
  File "/Users/xiaotingzhou/miniconda3/lib/python3.11/site-packages/ipykernel/ipkernel.py", line 770, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(

KeyboardInterrupt: 


使用设备: cpu


## 2. 数据加载和预处理

我们使用WikiText-2数据集，这是一个常用的语言建模基准数据集。该数据集包含从维基百科文章中提取的文本，适合训练语言模型。

In [ ]:
# 使用本地WikiText token文件
print('正在加载本地WikiText token文件...')

# 指定本地token文件路径
train_token_path = '/Users/xiaotingzhou/Documents/Lectures/ML_DL/LSTM/wiki.train.tokens'
valid_token_path = '/Users/xiaotingzhou/Documents/Lectures/ML_DL/LSTM/wiki.valid.tokens'
test_token_path = '/Users/xiaotingzhou/Documents/Lectures/ML_DL/LSTM/wiki.test.tokens'

def check_file_exists(file_path):
    """检查文件是否存在"""
    if os.path.exists(file_path):
        print(f'✓ 文件存在: {file_path}')
        return True
    else:
        print(f'✗ 文件不存在: {file_path}')
        return False

# 检查所有数据文件
all_files_exist = True
for path, name in [(train_token_path, '训练数据'),
                   (valid_token_path, '验证数据'),
                   (test_token_path, '测试数据')]:
    if check_file_exists(path):
        # 显示文件大小信息
        file_size = os.path.getsize(path) / (1024 * 1024)  # MB
        with open(path, 'r', encoding='utf-8') as f:
            line_count = sum(1 for _ in f)
        print(f'  {name}: {file_size:.2f} MB, {line_count:,} 行')
    else:
        all_files_exist = False

if not all_files_exist:
    print('\n警告: 某些数据文件不存在！请确保文件路径正确。')
else:
    print('\n所有数据文件检查完成，准备加载数据...')

In [ ]:
# 加载本地token文件数据
def load_token_file(file_path, max_lines=None):
    """
    加载token文件并进行预处理

    Args:
        file_path: 文件路径
        max_lines: 最大行数限制（用于快速测试）

    Returns:
        处理后的文本字符串
    """
    lines = []
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            for i, line in enumerate(f):
                if max_lines and i >= max_lines:
                    break

                line = line.strip()
                # 过滤掉标题行、空行和太短的行
                if (line and
                    not line.startswith('=') and
                    not line.startswith('@') and
                    len(line) > 5 and  # 至少5个字符
                    any(c.isalpha() for c in line)):  # 包含字母
                    lines.append(line)

        # 合并所有行为一个文本，用空格分隔
        text = ' '.join(lines)
        print(f'从 {file_path} 加载了 {len(lines)} 行有效文本，总长度: {len(text):,} 字符')
        return text

    except Exception as e:
        print(f'加载 {file_path} 时出错: {e}')
        return ""

# 为了演示和快速测试，我们可以限制数据量
USE_FULL_DATASET = True  # 设置为False可以使用较小的数据集进行快速测试
MAX_LINES_LIMIT = 5000 if not USE_FULL_DATASET else None

# 加载数据
print('正在加载训练数据...')
train_text = load_token_file(train_token_path, MAX_LINES_LIMIT)

print('正在加载验证数据...')
valid_text = load_token_file(valid_token_path, MAX_LINES_LIMIT // 5 if MAX_LINES_LIMIT else None)

print('正在加载测试数据...')
test_text = load_token_file(test_token_path, MAX_LINES_LIMIT // 5 if MAX_LINES_LIMIT else None)

print(f'\n数据加载完成:')
print(f'训练集长度: {len(train_text):,} 字符')
print(f'验证集长度: {len(valid_text):,} 字符')
print(f'测试集长度: {len(test_text):,} 字符')

# 显示数据示例
print(f'\n训练数据示例 (前500字符):')
print(train_text[:500])
print('...')

# 简单的数据质量检查
def check_data_quality(text, name):
    """检查数据质量"""
    if not text:
        print(f'警告: {name} 数据为空!')
        return False

    words = text.split()
    print(f'{name} 统计:')
    print(f'  总单词数: {len(words):,}')
    print(f'  唯一单词数: {len(set(words)):,}')
    print(f'  平均单词长度: {np.mean([len(w) for w in words[:1000]]):.1f}')  # 只统计前1000个词
    return True

# 检查所有数据集
for text, name in [(train_text, '训练集'), (valid_text, '验证集'), (test_text, '测试集')]:
    check_data_quality(text, name)
    print()

In [ ]:
# 文本预处理和词汇表构建
class TextProcessor:
    """文本处理器，用于构建词汇表和进行文本-数字转换"""

    def __init__(self, min_freq=2):
        self.min_freq = min_freq
        self.word_to_idx = {}
        self.idx_to_word = {}
        self.vocab_size = 0

        # 特殊标记
        self.pad_token = '<pad>'
        self.unk_token = '<unk>'
        self.eos_token = '<eos>'

    def tokenize(self, text):
        """简单的分词函数"""
        # 替换换行符为特殊标记
        text = text.replace('\n', ' <eos> ')
        # 简单按空格分词
        tokens = text.lower().split()
        return tokens

    def build_vocab(self, texts):
        """构建词汇表"""
        # 统计词频
        word_freq = Counter()
        for text in texts:
            tokens = self.tokenize(text)
            word_freq.update(tokens)

        # 添加特殊标记
        vocab = [self.pad_token, self.unk_token, self.eos_token]

        # 添加高频词
        for word, freq in word_freq.most_common():
            if freq >= self.min_freq:
                vocab.append(word)

        # 构建词汇表映射
        self.word_to_idx = {word: idx for idx, word in enumerate(vocab)}
        self.idx_to_word = {idx: word for idx, word in enumerate(vocab)}
        self.vocab_size = len(vocab)

        print(f'词汇表大小: {self.vocab_size}')
        print(f'最高频的10个词: {vocab[3:13]}')

    def encode(self, text):
        """将文本编码为数字序列"""
        tokens = self.tokenize(text)
        indices = []
        for token in tokens:
            if token in self.word_to_idx:
                indices.append(self.word_to_idx[token])
            else:
                indices.append(self.word_to_idx[self.unk_token])
        return indices

    def decode(self, indices):
        """将数字序列解码为文本"""
        words = [self.idx_to_word[idx] for idx in indices]
        return ' '.join(words)

# 创建文本处理器并构建词汇表
processor = TextProcessor(min_freq=3)
processor.build_vocab([train_text])

# 编码所有数据
train_indices = processor.encode(train_text)
valid_indices = processor.encode(valid_text)
test_indices = processor.encode(test_text)

print(f'\n编码后序列长度:')
print(f'训练集: {len(train_indices)} tokens')
print(f'验证集: {len(valid_indices)} tokens')
print(f'测试集: {len(test_indices)} tokens')

## 3. 探索性数据分析

在训练模型之前，让我们分析数据集的特征，了解词汇分布、序列长度等统计信息。

In [ ]:
# 数据统计分析
def analyze_data(indices, processor, dataset_name):
    """分析数据集统计信息"""
    print(f'\n=== {dataset_name} 数据分析 ===')
    print(f'总token数量: {len(indices):,}')

    # 词频分析
    word_freq = Counter(indices)
    print(f'唯一词汇数: {len(word_freq)}')

    # 最常见的词
    print('\n最常见的10个词:')
    for idx, freq in word_freq.most_common(10):
        word = processor.idx_to_word[idx]
        print(f'  {word}: {freq:,} 次 ({freq/len(indices)*100:.2f}%)')

    return word_freq

# 分析训练数据
train_freq = analyze_data(train_indices, processor, '训练集')
valid_freq = analyze_data(valid_indices, processor, '验证集')

In [ ]:
# 可视化数据分布
plt.figure(figsize=(15, 10))

# 1. 词频分布
plt.subplot(2, 3, 1)
freq_values = list(train_freq.values())
plt.hist(freq_values, bins=50, alpha=0.7, color='skyblue')
plt.xlabel('词频')
plt.ylabel('词汇数量')
plt.title('词频分布直方图')
plt.yscale('log')
plt.xscale('log')

# 2. 前50个最常见词的频率
plt.subplot(2, 3, 2)
top_50_freq = [freq for _, freq in train_freq.most_common(50)]
plt.plot(range(1, 51), top_50_freq, 'o-', color='red')
plt.xlabel('词汇排名')
plt.ylabel('频率')
plt.title('前50个高频词分布')
plt.yscale('log')

# 3. 累积频率分布
plt.subplot(2, 3, 3)
sorted_freqs = sorted(freq_values, reverse=True)
cumsum_freqs = np.cumsum(sorted_freqs)
coverage = cumsum_freqs / sum(freq_values)
plt.plot(range(1, len(coverage)+1), coverage)
plt.xlabel('词汇数量')
plt.ylabel('覆盖率')
plt.title('词汇覆盖率曲线')
plt.axhline(y=0.8, color='red', linestyle='--', alpha=0.7, label='80%覆盖率')
plt.axhline(y=0.9, color='orange', linestyle='--', alpha=0.7, label='90%覆盖率')
plt.legend()
plt.xscale('log')

# 4. 句子长度分布（以<eos>为分隔符）
plt.subplot(2, 3, 4)
eos_idx = processor.word_to_idx[processor.eos_token]
sentence_lengths = []
current_length = 0
for idx in train_indices[:50000]:  # 只分析前50k个token以提高速度
    if idx == eos_idx:
        if current_length > 0:
            sentence_lengths.append(current_length)
        current_length = 0
    else:
        current_length += 1

plt.hist(sentence_lengths, bins=50, alpha=0.7, color='green')
plt.xlabel('句子长度（单词数）')
plt.ylabel('句子数量')
plt.title('句子长度分布')
plt.axvline(x=np.mean(sentence_lengths), color='red', linestyle='--', label=f'平均长度: {np.mean(sentence_lengths):.1f}')
plt.legend()

# 5. 数据集大小对比
plt.subplot(2, 3, 5)
dataset_sizes = [len(train_indices), len(valid_indices), len(test_indices)]
dataset_names = ['训练集', '验证集', '测试集']
colors = ['blue', 'orange', 'green']
bars = plt.bar(dataset_names, dataset_sizes, color=colors, alpha=0.7)
plt.ylabel('Token数量')
plt.title('数据集大小对比')
# 在柱状图上添加数值标签
for bar, size in zip(bars, dataset_sizes):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(dataset_sizes)*0.01,
             f'{size:,}', ha='center', va='bottom')

# 6. 词汇表统计
plt.subplot(2, 3, 6)
vocab_stats = {
    '总词汇量': processor.vocab_size,
    '训练集唯一词': len(train_freq),
    '验证集唯一词': len(valid_freq)
}
bars = plt.bar(vocab_stats.keys(), vocab_stats.values(), color=['purple', 'cyan', 'pink'], alpha=0.7)
plt.ylabel('词汇数量')
plt.title('词汇表统计')
plt.xticks(rotation=45)
for bar, value in zip(bars, vocab_stats.values()):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(vocab_stats.values())*0.01,
             f'{value:,}', ha='center', va='bottom')

plt.tight_layout()
plt.show()

# 打印一些统计信息
print(f'\n=== 数据集总体统计 ===')
print(f'平均句子长度: {np.mean(sentence_lengths):.1f} 个单词')
print(f'句子长度标准差: {np.std(sentence_lengths):.1f} 个单词')
print(f'最长句子: {max(sentence_lengths)} 个单词')
print(f'最短句子: {min(sentence_lengths)} 个单词')

# 计算需要多少个词汇能覆盖80%和90%的数据
vocab_80 = np.where(coverage >= 0.8)[0][0] + 1
vocab_90 = np.where(coverage >= 0.9)[0][0] + 1
print(f'\n覆盖80%数据需要的词汇量: {vocab_80:,}')
print(f'覆盖90%数据需要的词汇量: {vocab_90:,}')

## 4. 数据集类和数据加载器

创建PyTorch数据集类来处理序列数据的批量加载。

In [ ]:
# 数据集类和数据加载器 - 优化版本
class WikiTextDataset(Dataset):
    """WikiText数据集类 - 优化版本"""

    def __init__(self, indices, seq_length=30):  # 减小序列长度
        self.indices = indices
        self.seq_length = seq_length

        # 创建序列对，每个序列包含seq_length个词，目标是下一个词
        self.sequences = []
        self.targets = []

        print(f'正在创建序列，序列长度: {seq_length}...')

        # 限制数据量以确保可以在合理时间内完成训练
        max_sequences = 50000  # 限制最大序列数量
        step_size = max(1, (len(indices) - seq_length) // max_sequences)  # 采样步长

        for i in range(0, len(indices) - seq_length, step_size):
            if len(self.sequences) >= max_sequences:
                break
            sequence = indices[i:i + seq_length]
            target = indices[i + seq_length]
            self.sequences.append(sequence)
            self.targets.append(target)

        print(f'创建了 {len(self.sequences):,} 个训练序列')

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        return torch.tensor(self.sequences[idx], dtype=torch.long), torch.tensor(self.targets[idx], dtype=torch.long)

# 超参数设置 - 优化版本
SEQUENCE_LENGTH = 30      # 减小序列长度以节省内存
BATCH_SIZE = 32          # 保持合理的批次大小
NUM_WORKERS = 0          # Windows系统建议设为0

# 创建数据集
print('正在创建数据集...')
try:
    train_dataset = WikiTextDataset(train_indices, SEQUENCE_LENGTH)
    valid_dataset = WikiTextDataset(valid_indices, SEQUENCE_LENGTH)
    test_dataset = WikiTextDataset(test_indices, SEQUENCE_LENGTH)

    print(f'数据集创建成功:')
    print(f'训练样本数: {len(train_dataset):,}')
    print(f'验证样本数: {len(valid_dataset):,}')
    print(f'测试样本数: {len(test_dataset):,}')

    # 创建数据加载器
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
    valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

    print(f'\n数据加载器创建成功:')
    print(f'训练批次数: {len(train_loader)}')
    print(f'验证批次数: {len(valid_loader)}')
    print(f'测试批次数: {len(test_loader)}')

    # 查看一个批次的数据
    try:
        sample_batch = next(iter(train_loader))
        sequences, targets = sample_batch
        print(f'\n批次数据验证成功:')
        print(f'序列形状: {sequences.shape}')
        print(f'目标形状: {targets.shape}')

        print('\n第一个样本示例:')
        sample_sequence = sequences[0].tolist()
        sample_target = targets[0].item()

        print(f'输入序列: {processor.decode(sample_sequence)}')
        print(f'目标词: {processor.idx_to_word[sample_target]}')

    except Exception as e:
        print(f'批次数据验证出错: {e}')

except Exception as e:
    print(f'创建数据集时出错: {e}')
    print('这可能是由于内存不足或数据处理问题')

    # 提供备用的小数据集
    print('创建小型演示数据集...')
    small_train_indices = train_indices[:10000]
    small_valid_indices = valid_indices[:2000]
    small_test_indices = test_indices[:2000]

    train_dataset = WikiTextDataset(small_train_indices, SEQUENCE_LENGTH)
    valid_dataset = WikiTextDataset(small_valid_indices, SEQUENCE_LENGTH)
    test_dataset = WikiTextDataset(small_test_indices, SEQUENCE_LENGTH)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
    valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

    print(f'小型数据集创建成功: 训练{len(train_dataset)}, 验证{len(valid_dataset)}, 测试{len(test_dataset)}')

## 5. LSTM模型架构实现

实现一个完整的LSTM语言模型，包括嵌入层、LSTM层、Dropout和输出层。

In [ ]:
# LSTM模型架构实现 - 优化版本
class LSTMLanguageModel(nn.Module):
    """LSTM语言模型 - 优化版本"""

    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_layers, dropout_rate=0.3):
        super(LSTMLanguageModel, self).__init__()

        self.vocab_size = vocab_size
        self.embedding_dim = embedding_dim
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.dropout_rate = dropout_rate

        # 词嵌入层
        self.embedding = nn.Embedding(vocab_size, embedding_dim)

        # LSTM层
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers,
                           batch_first=True, dropout=dropout_rate if num_layers > 1 else 0)

        # Dropout层
        self.dropout = nn.Dropout(dropout_rate)

        # 输出全连接层
        self.fc = nn.Linear(hidden_dim, vocab_size)

        # 权重初始化
        self.init_weights()

    def init_weights(self):
        """初始化模型权重"""
        init_range = 0.1
        self.embedding.weight.data.uniform_(-init_range, init_range)
        self.fc.weight.data.uniform_(-init_range, init_range)
        self.fc.bias.data.zero_()

        # LSTM权重初始化
        for name, param in self.lstm.named_parameters():
            if 'weight_ih' in name:
                torch.nn.init.xavier_uniform_(param.data)
            elif 'weight_hh' in name:
                torch.nn.init.orthogonal_(param.data)
            elif 'bias' in name:
                param.data.fill_(0)
                # 设置遗忘门偏置为1（帮助记忆长期依赖）
                n = param.size(0)
                param.data[n//4:n//2].fill_(1.0)

    def forward(self, x, hidden=None):
        """前向传播"""
        batch_size, seq_length = x.size()

        # 词嵌入
        embedded = self.embedding(x)  # (batch_size, seq_length, embedding_dim)
        embedded = self.dropout(embedded)

        # LSTM层
        lstm_out, hidden = self.lstm(embedded, hidden)  # (batch_size, seq_length, hidden_dim)
        lstm_out = self.dropout(lstm_out)

        # 输出层
        output = self.fc(lstm_out)  # (batch_size, seq_length, vocab_size)

        return output, hidden

    def init_hidden(self, batch_size, device):
        """初始化隐藏状态"""
        h0 = torch.zeros(self.num_layers, batch_size, self.hidden_dim).to(device)
        c0 = torch.zeros(self.num_layers, batch_size, self.hidden_dim).to(device)
        return (h0, c0)

# 模型超参数 - 优化版本（较小的模型以确保训练成功）
EMBEDDING_DIM = 128      # 减小词嵌入维度
HIDDEN_DIM = 256         # 减小LSTM隐藏层维度
NUM_LAYERS = 2           # 保持2层
DROPOUT_RATE = 0.3       # Dropout率

print(f'模型配置:')
print(f'词汇表大小: {processor.vocab_size:,}')
print(f'嵌入维度: {EMBEDDING_DIM}')
print(f'隐藏层维度: {HIDDEN_DIM}')
print(f'LSTM层数: {NUM_LAYERS}')
print(f'Dropout率: {DROPOUT_RATE}')
print(f'序列长度: {SEQUENCE_LENGTH}')

# 创建模型
try:
    model = LSTMLanguageModel(
        vocab_size=processor.vocab_size,
        embedding_dim=EMBEDDING_DIM,
        hidden_dim=HIDDEN_DIM,
        num_layers=NUM_LAYERS,
        dropout_rate=DROPOUT_RATE
    ).to(device)

    # 计算模型参数数量
    def count_parameters(model):
        return sum(p.numel() for p in model.parameters() if p.requires_grad)

    total_params = count_parameters(model)
    print(f'\n模型创建成功!')
    print(f'模型总参数数量: {total_params:,}')
    print(f'模型大小估计: {total_params * 4 / 1024 / 1024:.1f} MB (FP32)')

    # 打印模型架构摘要
    print(f'\n模型架构摘要:')
    print(f'├─ Embedding: {processor.vocab_size:,} × {EMBEDDING_DIM} = {processor.vocab_size * EMBEDDING_DIM:,} 参数')

    # LSTM参数计算
    lstm_params = 4 * (EMBEDDING_DIM * HIDDEN_DIM + HIDDEN_DIM * HIDDEN_DIM + HIDDEN_DIM) * NUM_LAYERS
    if NUM_LAYERS > 1:
        lstm_params += 4 * (HIDDEN_DIM * HIDDEN_DIM + HIDDEN_DIM * HIDDEN_DIM + HIDDEN_DIM) * (NUM_LAYERS - 1)
    print(f'├─ LSTM: ~{lstm_params:,} 参数')

    fc_params = HIDDEN_DIM * processor.vocab_size + processor.vocab_size
    print(f'└─ Linear: {fc_params:,} 参数')

    # 测试模型前向传播
    model.eval()
    with torch.no_grad():
        try:
            # 使用之前创建的样本数据进行测试
            test_input = sequences[:2].to(device)  # 取2个样本测试
            print(f'\n测试输入形状: {test_input.shape}')

            test_output, test_hidden = model(test_input)
            print(f'测试输出形状: {test_output.shape}')  # 应该是 (2, seq_length, vocab_size)
            print(f'隐藏状态形状: h={test_hidden[0].shape}, c={test_hidden[1].shape}')
            print('模型前向传播测试通过!')

        except Exception as e:
            print(f'模型前向传播测试失败: {e}')

    model.train()

except Exception as e:
    print(f'模型创建失败: {e}')
    print('这可能是由于内存不足或其他配置问题')

    # 尝试创建更小的模型
    print('\n尝试创建更小的模型...')
    model = LSTMLanguageModel(
        vocab_size=min(processor.vocab_size, 5000),  # 限制词汇表大小
        embedding_dim=64,    # 更小的嵌入维度
        hidden_dim=128,      # 更小的隐藏层维度
        num_layers=1,        # 单层LSTM
        dropout_rate=0.3
    ).to(device)

    total_params = count_parameters(model)
    print(f'小型模型参数数量: {total_params:,}')

print('\n模型准备就绪!')

## 6. 训练配置和损失函数

设置训练相关的配置，包括损失函数、优化器和评估指标。

In [ ]:
# 训练配置和损失函数 - 优化版本
# 训练超参数 - 优化以确保训练成功
LEARNING_RATE = 0.002    # 稍微提高学习率
NUM_EPOCHS = 15          # 减少训练轮数
GRADIENT_CLIP = 1.0      # 梯度裁剪阈值
PRINT_EVERY = 50         # 减少打印频率

print(f'训练配置:')
print(f'学习率: {LEARNING_RATE}')
print(f'训练轮数: {NUM_EPOCHS}')
print(f'梯度裁剪阈值: {GRADIENT_CLIP}')
print(f'设备: {device}')

# 损失函数和优化器
try:
    criterion = nn.CrossEntropyLoss(ignore_index=processor.word_to_idx.get(processor.pad_token, 0))
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-5)

    # 学习率调度器 - 使用更温和的调度策略
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.7, patience=3, verbose=True, min_lr=1e-6
    )

    print(f'优化器: {type(optimizer).__name__}')
    print(f'学习率调度器: {type(scheduler).__name__}')
    print('训练组件初始化成功!')

except Exception as e:
    print(f'训练组件初始化失败: {e}')

def calculate_perplexity(loss):
    """计算困惑度，添加数值稳定性检查"""
    try:
        if loss > 50:  # 避免计算过大的指数
            return float('inf')
        return math.exp(min(loss, 50))
    except (OverflowError, ValueError):
        return float('inf')

def evaluate_model(model, data_loader, criterion, device, max_batches=None):
    """
    评估模型性能，添加错误处理和内存管理

    Args:
        model: 模型
        data_loader: 数据加载器
        criterion: 损失函数
        device: 设备
        max_batches: 最大评估批次数（用于加速评估）
    """
    model.eval()
    total_loss = 0
    total_samples = 0
    evaluated_batches = 0

    try:
        with torch.no_grad():
            for batch_idx, (sequences, targets) in enumerate(tqdm(data_loader, desc='评估中', leave=False)):
                if max_batches and batch_idx >= max_batches:
                    break

                try:
                    sequences, targets = sequences.to(device), targets.to(device)

                    # 前向传播
                    outputs, _ = model(sequences)

                    # 计算损失 - 只使用最后一个时间步的输出
                    loss = criterion(outputs[:, -1, :], targets)

                    # 检查损失是否为有效数值
                    if not torch.isfinite(loss):
                        print(f'警告: 发现无效损失值 {loss.item()}, 跳过此批次')
                        continue

                    total_loss += loss.item() * sequences.size(0)
                    total_samples += sequences.size(0)
                    evaluated_batches += 1

                except Exception as e:
                    print(f'评估批次 {batch_idx} 时出错: {e}')
                    continue

        if total_samples == 0:
            print('警告: 没有成功评估任何样本')
            return float('inf'), float('inf')

        avg_loss = total_loss / total_samples
        perplexity = calculate_perplexity(avg_loss)

        print(f'评估完成: {evaluated_batches} 批次, {total_samples} 样本')
        return avg_loss, perplexity

    except Exception as e:
        print(f'评估过程中发生错误: {e}')
        return float('inf'), float('inf')

    finally:
        model.train()  # 确保模型回到训练模式

print('评估函数定义完成!')

## 7. 训练循环实现

实现完整的训练循环，包括梯度裁剪、进度跟踪和性能监控。

In [ ]:
# 训练循环实现 - 鲁棒版本
# 训练历史记录
train_losses = []
valid_losses = []
train_perplexities = []
valid_perplexities = []
learning_rates = []

# 最佳模型追踪
best_valid_loss = float('inf')
best_model_state = None
patience_counter = 0
PATIENCE = 5  # 早停耐心值

# 训练统计
total_train_time = 0
successful_epochs = 0

print('开始训练...')
print('='*80)

try:
    import time
    training_start_time = time.time()

    for epoch in range(NUM_EPOCHS):
        epoch_start_time = time.time()

        # 训练阶段
        model.train()
        total_train_loss = 0
        num_batches = 0
        successful_batches = 0

        # 创建进度条
        pbar = tqdm(train_loader, desc=f'训练 Epoch {epoch+1}/{NUM_EPOCHS}')

        try:
            for batch_idx, (sequences, targets) in enumerate(pbar):
                try:
                    sequences, targets = sequences.to(device), targets.to(device)

                    # 清零梯度
                    optimizer.zero_grad()

                    # 前向传播
                    outputs, _ = model(sequences)

                    # 计算损失 - 只使用最后一个时间步的输出预测下一个词
                    loss = criterion(outputs[:, -1, :], targets)

                    # 检查损失是否为有效数值
                    if not torch.isfinite(loss):
                        print(f'\n警告: Epoch {epoch+1}, Batch {batch_idx} 损失为无效值 {loss.item()}')
                        continue

                    # 反向传播
                    loss.backward()

                    # 检查梯度
                    grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP)
                    if not torch.isfinite(grad_norm):
                        print(f'\n警告: Epoch {epoch+1}, Batch {batch_idx} 梯度异常')
                        optimizer.zero_grad()
                        continue

                    # 更新参数
                    optimizer.step()

                    # 累积损失
                    total_train_loss += loss.item()
                    num_batches += 1
                    successful_batches += 1

                    # 更新进度条
                    if num_batches > 0:
                        avg_loss = total_train_loss / num_batches
                        pbar.set_postfix({
                            'Loss': f'{avg_loss:.4f}',
                            'PPL': f'{calculate_perplexity(avg_loss):.2f}',
                            'LR': f'{optimizer.param_groups[0]["lr"]:.6f}',
                            'Grad': f'{grad_norm:.3f}'
                        })

                    # 定期打印详细信息
                    if (batch_idx + 1) % PRINT_EVERY == 0:
                        current_lr = optimizer.param_groups[0]['lr']
                        print(f'\nEpoch {epoch+1}, Batch {batch_idx+1}/{len(train_loader)}:')
                        print(f'  训练损失: {avg_loss:.4f}')
                        print(f'  训练困惑度: {calculate_perplexity(avg_loss):.2f}')
                        print(f'  学习率: {current_lr:.6f}')
                        print(f'  成功批次: {successful_batches}/{num_batches}')

                except Exception as e:
                    print(f'\n批次 {batch_idx} 训练出错: {e}')
                    continue

            # 检查是否有成功的批次
            if num_batches == 0:
                print(f'Epoch {epoch+1}: 没有成功的训练批次，跳过此轮')
                continue

            # 计算平均训练损失
            avg_train_loss = total_train_loss / num_batches
            train_perplexity = calculate_perplexity(avg_train_loss)

            # 验证阶段
            print(f'\n正在验证 Epoch {epoch+1}...')
            valid_loss, valid_perplexity = evaluate_model(
                model, valid_loader, criterion, device, max_batches=50  # 限制验证批次以节省时间
            )

            # 检查验证结果是否有效
            if not math.isfinite(valid_loss):
                print(f'Epoch {epoch+1}: 验证损失无效，使用训练损失作为替代')
                valid_loss = avg_train_loss * 1.2  # 稍微提高作为验证损失的估计
                valid_perplexity = calculate_perplexity(valid_loss)

            # 更新学习率
            scheduler.step(valid_loss)
            current_lr = optimizer.param_groups[0]['lr']

            # 记录历史
            train_losses.append(avg_train_loss)
            valid_losses.append(valid_loss)
            train_perplexities.append(train_perplexity)
            valid_perplexities.append(valid_perplexity)
            learning_rates.append(current_lr)

            # 计算训练时间
            epoch_time = time.time() - epoch_start_time
            total_train_time += epoch_time
            successful_epochs += 1

            # 打印epoch结果
            print(f'\nEpoch {epoch+1} 完成! (用时: {epoch_time:.1f}s)')
            print(f'训练损失: {avg_train_loss:.4f} | 训练困惑度: {train_perplexity:.2f}')
            print(f'验证损失: {valid_loss:.4f} | 验证困惑度: {valid_perplexity:.2f}')
            print(f'学习率: {current_lr:.6f}')
            print(f'成功批次率: {successful_batches}/{len(train_loader)} ({successful_batches/len(train_loader)*100:.1f}%)')

            # 保存最佳模型
            if valid_loss < best_valid_loss:
                best_valid_loss = valid_loss
                best_model_state = model.state_dict().copy()
                patience_counter = 0
                print(f'新的最佳模型! 验证损失: {valid_loss:.4f}')

                # 保存最佳模型到文件
                try:
                    torch.save({
                        'epoch': epoch + 1,
                        'model_state_dict': best_model_state,
                        'optimizer_state_dict': optimizer.state_dict(),
                        'loss': best_valid_loss,
                        'vocab_size': processor.vocab_size,
                        'embedding_dim': EMBEDDING_DIM,
                        'hidden_dim': HIDDEN_DIM,
                        'num_layers': NUM_LAYERS
                    }, 'best_lstm_model.pth')
                    print('最佳模型已保存到 best_lstm_model.pth')
                except Exception as e:
                    print(f'模型保存失败: {e}')
            else:
                patience_counter += 1
                print(f'验证损失未改善，耐心计数: {patience_counter}/{PATIENCE}')

            # 早停检查
            if patience_counter >= PATIENCE:
                print(f'\n早停触发! 验证损失连续{PATIENCE}轮未改善')
                break

            print('='*80)

        except Exception as e:
            print(f'Epoch {epoch+1} 训练过程中出现严重错误: {e}')
            print('尝试继续下一个epoch...')
            continue

    training_end_time = time.time()
    total_training_time = training_end_time - training_start_time

    print('\n训练完成!')
    print(f'总训练时间: {total_training_time:.1f}s ({total_training_time/60:.1f}min)')
    print(f'成功完成的轮次: {successful_epochs}/{NUM_EPOCHS}')
    print(f'最佳验证损失: {best_valid_loss:.4f}')
    print(f'最佳验证困惑度: {calculate_perplexity(best_valid_loss):.2f}')

    # 加载最佳模型
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
        print('已加载最佳模型权重')
    else:
        print('警告: 没有找到有效的最佳模型')

except KeyboardInterrupt:
    print('\n训练被用户中断')
    print('保存当前模型状态...')
    try:
        torch.save({
            'epoch': epoch + 1 if 'epoch' in locals() else 0,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'train_losses': train_losses,
            'valid_losses': valid_losses
        }, 'interrupted_model.pth')
        print('当前模型已保存到 interrupted_model.pth')
    except:
        print('模型保存失败')

except Exception as e:
    print(f'\n训练过程中发生未预期的错误: {e}')
    print('尝试保存当前模型状态...')
    try:
        torch.save(model.state_dict(), 'error_model.pth')
        print('模型已保存到 error_model.pth')
    except:
        pass

# 清理GPU内存
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print('GPU内存已清理')

print('\n训练阶段结束')

## 8. 模型评估和可视化

评估训练好的模型并可视化训练过程。

In [ ]:
# 在测试集上评估最终模型
print('正在测试集上评估模型...')
test_loss, test_perplexity = evaluate_model(model, test_loader, criterion, device)

print(f'\n=== 最终模型性能 ===')
print(f'测试损失: {test_loss:.4f}')
print(f'测试困惑度: {test_perplexity:.2f}')
print(f'最佳验证损失: {best_valid_loss:.4f}')
print(f'最佳验证困惑度: {calculate_perplexity(best_valid_loss):.2f}')

In [ ]:
# 可视化训练过程
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

epochs = range(1, len(train_losses) + 1)

# 1. 损失曲线
axes[0, 0].plot(epochs, train_losses, 'b-', label='训练损失', linewidth=2)
axes[0, 0].plot(epochs, valid_losses, 'r-', label='验证损失', linewidth=2)
axes[0, 0].set_xlabel('训练轮次')
axes[0, 0].set_ylabel('损失值')
axes[0, 0].set_title('训练和验证损失')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# 2. 困惑度曲线
axes[0, 1].plot(epochs, train_perplexities, 'b-', label='训练困惑度', linewidth=2)
axes[0, 1].plot(epochs, valid_perplexities, 'r-', label='验证困惑度', linewidth=2)
axes[0, 1].set_xlabel('训练轮次')
axes[0, 1].set_ylabel('困惑度')
axes[0, 1].set_title('训练和验证困惑度')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].set_yscale('log')

# 3. 学习率变化
axes[1, 0].plot(epochs, learning_rates, 'g-', linewidth=2)
axes[1, 0].set_xlabel('训练轮次')
axes[1, 0].set_ylabel('学习率')
axes[1, 0].set_title('学习率调度')
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].set_yscale('log')

# 4. 训练进度总结
axes[1, 1].axis('off')
summary_text = f"""
训练总结:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
• 总训练轮次: {len(train_losses)}
• 模型参数量: {total_params:,}
• 词汇表大小: {processor.vocab_size:,}
• 序列长度: {SEQUENCE_LENGTH}
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
最终性能:
• 最佳验证损失: {best_valid_loss:.4f}
• 最佳验证困惑度: {calculate_perplexity(best_valid_loss):.2f}
• 测试损失: {test_loss:.4f}
• 测试困惑度: {test_perplexity:.2f}
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
训练配置:
• 批次大小: {BATCH_SIZE}
• 初始学习率: {LEARNING_RATE}
• 嵌入维度: {EMBEDDING_DIM}
• 隐藏层维度: {HIDDEN_DIM}
• LSTM层数: {NUM_LAYERS}
• Dropout率: {DROPOUT_RATE}
"""

axes[1, 1].text(0.05, 0.95, summary_text, transform=axes[1, 1].transAxes,
                fontsize=10, verticalalignment='top', fontfamily='monospace',
                bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.8))

plt.tight_layout()
plt.show()

# 保存训练历史
training_history = {
    'train_losses': train_losses,
    'valid_losses': valid_losses,
    'train_perplexities': train_perplexities,
    'valid_perplexities': valid_perplexities,
    'learning_rates': learning_rates,
    'best_valid_loss': best_valid_loss,
    'test_loss': test_loss,
    'test_perplexity': test_perplexity
}

# 保存训练历史到文件
with open('training_history.pkl', 'wb') as f:
    pickle.dump(training_history, f)

print('\n训练历史已保存到 training_history.pkl')

## 9. 文本生成示例

使用训练好的模型生成新的文本内容。

In [ ]:
def generate_text(model, processor, seed_text, max_length=100, temperature=1.0, top_k=50):
    """
    生成文本

    Args:
        model: 训练好的模型
        processor: 文本处理器
        seed_text: 种子文本
        max_length: 最大生成长度
        temperature: 温度参数，控制随机性
        top_k: Top-K采样参数
    """
    model.eval()

    # 编码种子文本
    tokens = processor.encode(seed_text)

    # 确保种子文本长度足够
    if len(tokens) < SEQUENCE_LENGTH:
        # 用填充标记补齐
        pad_idx = processor.word_to_idx[processor.pad_token]
        tokens = [pad_idx] * (SEQUENCE_LENGTH - len(tokens)) + tokens
    else:
        tokens = tokens[-SEQUENCE_LENGTH:]  # 只取最后SEQUENCE_LENGTH个tokens

    generated = tokens.copy()

    with torch.no_grad():
        for _ in range(max_length):
            # 准备输入
            input_seq = torch.tensor([tokens[-SEQUENCE_LENGTH:]], dtype=torch.long).to(device)

            # 前向传播
            outputs, _ = model(input_seq)

            # 获取最后一个时间步的输出
            logits = outputs[0, -1, :] / temperature

            # Top-K采样
            if top_k > 0:
                top_k_logits, top_k_indices = torch.topk(logits, top_k)
                # 将非top-k的概率设为负无穷
                logits.fill_(-float('inf'))
                logits.scatter_(0, top_k_indices, top_k_logits)

            # 计算概率并采样
            probs = F.softmax(logits, dim=0)
            next_token = torch.multinomial(probs, 1).item()

            # 添加到序列
            tokens.append(next_token)
            generated.append(next_token)

            # 如果生成了结束标记，停止生成
            if next_token == processor.word_to_idx[processor.eos_token]:
                break

    # 解码生成的文本
    generated_text = processor.decode(generated)
    return generated_text

# 测试文本生成
print('=== 文本生成示例 ===')
print()

# 不同的种子文本
seed_texts = [
    "The history of",
    "In the year",
    "The king of",
    "During the war",
    "The scientific discovery"
]

# 不同的温度设置
temperatures = [0.5, 1.0, 1.5]

for i, seed in enumerate(seed_texts, 1):
    print(f'{i}. 种子文本: "{seed}"')
    print('-' * 60)

    for temp in temperatures:
        generated = generate_text(model, processor, seed, max_length=50,
                                temperature=temp, top_k=40)
        print(f'   温度={temp}: {generated}')
    print()

# 更长的文本生成示例
print('\n=== 长文本生成示例 ===')
print('-' * 80)
long_seed = "The ancient civilization"
long_text = generate_text(model, processor, long_seed, max_length=150,
                         temperature=0.8, top_k=30)
print(f'种子: "{long_seed}"')
print(f'生成: {long_text}')

## 10. 高级优化策略

展示一些高级的训练优化技术，包括更复杂的学习率调度、正则化技术等。

In [ ]:
# 高级模型类，包含更多优化技术
class AdvancedLSTMLanguageModel(nn.Module):
    """高级LSTM语言模型，包含更多优化技术"""

    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_layers,
                 dropout_rate=0.3, tie_weights=True, use_layer_norm=True):
        super(AdvancedLSTMLanguageModel, self).__init__()

        self.vocab_size = vocab_size
        self.embedding_dim = embedding_dim
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.dropout_rate = dropout_rate
        self.tie_weights = tie_weights
        self.use_layer_norm = use_layer_norm

        # 词嵌入层
        self.embedding = nn.Embedding(vocab_size, embedding_dim)

        # 输入dropout
        self.input_dropout = nn.Dropout(dropout_rate)

        # LSTM层
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers,
                           batch_first=True, dropout=dropout_rate if num_layers > 1 else 0)

        # 层归一化
        if use_layer_norm:
            self.layer_norm = nn.LayerNorm(hidden_dim)

        # 输出dropout
        self.output_dropout = nn.Dropout(dropout_rate)

        # 输出全连接层
        self.fc = nn.Linear(hidden_dim, vocab_size)

        # 权重绑定：输出层权重与嵌入层权重相同
        if tie_weights:
            if hidden_dim != embedding_dim:
                raise ValueError('使用权重绑定时，hidden_dim必须等于embedding_dim')
            self.fc.weight = self.embedding.weight

        # 权重初始化
        self.init_weights()

    def init_weights(self):
        """初始化模型权重"""
        init_range = 0.1

        # 嵌入层权重初始化
        nn.init.uniform_(self.embedding.weight, -init_range, init_range)

        # 输出层权重初始化（如果没有权重绑定）
        if not self.tie_weights:
            nn.init.uniform_(self.fc.weight, -init_range, init_range)
        nn.init.zeros_(self.fc.bias)

        # LSTM权重初始化
        for name, param in self.lstm.named_parameters():
            if 'weight_ih' in name:
                nn.init.xavier_uniform_(param)
            elif 'weight_hh' in name:
                nn.init.orthogonal_(param)
            elif 'bias' in name:
                nn.init.zeros_(param)
                # 设置遗忘门偏置为1
                n = param.size(0)
                param[n//4:n//2].fill_(1.0)

    def forward(self, x, hidden=None):
        """前向传播"""
        # 词嵌入和dropout
        embedded = self.embedding(x)
        embedded = self.input_dropout(embedded)

        # LSTM层
        lstm_out, hidden = self.lstm(embedded, hidden)

        # 层归一化
        if self.use_layer_norm:
            lstm_out = self.layer_norm(lstm_out)

        # 输出dropout
        lstm_out = self.output_dropout(lstm_out)

        # 输出层
        output = self.fc(lstm_out)

        return output, hidden

    def init_hidden(self, batch_size, device):
        """初始化隐藏状态"""
        h0 = torch.zeros(self.num_layers, batch_size, self.hidden_dim).to(device)
        c0 = torch.zeros(self.num_layers, batch_size, self.hidden_dim).to(device)
        return (h0, c0)

print('高级LSTM模型定义完成！')
print('\n新增特性:')
print('• 权重绑定 (Weight Tying)')
print('• 层归一化 (Layer Normalization)')
print('• 改进的权重初始化')
print('• 遗忘门偏置初始化为1')
print('• 分离的输入和输出dropout')

In [ ]:
# 高级训练器类
class AdvancedTrainer:
    """高级训练器，包含更多训练技巧"""

    def __init__(self, model, train_loader, valid_loader, device):
        self.model = model
        self.train_loader = train_loader
        self.valid_loader = valid_loader
        self.device = device

        # 训练配置
        self.criterion = nn.CrossEntropyLoss(ignore_index=processor.word_to_idx[processor.pad_token])

        # 使用AdamW优化器（带权重衰减的Adam）
        self.optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)

        # 余弦退火学习率调度器
        self.scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
            self.optimizer, T_0=5, T_mult=2, eta_min=1e-6
        )

        # 早停配置
        self.patience = 7
        self.min_delta = 1e-4
        self.best_loss = float('inf')
        self.patience_counter = 0
        self.best_model_state = None

        # 梯度累积
        self.accumulation_steps = 4

        # 训练历史
        self.history = {
            'train_losses': [],
            'valid_losses': [],
            'learning_rates': [],
            'grad_norms': []
        }

    def train_epoch(self, epoch):
        """训练一个epoch"""
        self.model.train()
        total_loss = 0
        num_batches = len(self.train_loader)
        grad_norms = []

        pbar = tqdm(self.train_loader, desc=f'训练 Epoch {epoch+1}')

        for batch_idx, (sequences, targets) in enumerate(pbar):
            sequences, targets = sequences.to(self.device), targets.to(self.device)

            # 前向传播
            outputs, _ = self.model(sequences)
            loss = self.criterion(outputs[:, -1, :], targets)

            # 梯度累积
            loss = loss / self.accumulation_steps
            loss.backward()

            if (batch_idx + 1) % self.accumulation_steps == 0:
                # 计算梯度范数
                grad_norm = torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
                grad_norms.append(grad_norm.item())

                # 更新参数
                self.optimizer.step()
                self.scheduler.step()
                self.optimizer.zero_grad()

            total_loss += loss.item() * self.accumulation_steps

            # 更新进度条
            avg_loss = total_loss / (batch_idx + 1)
            pbar.set_postfix({
                'Loss': f'{avg_loss:.4f}',
                'PPL': f'{math.exp(avg_loss):.2f}',
                'LR': f'{self.optimizer.param_groups[0]["lr"]:.6f}'
            })

        avg_loss = total_loss / num_batches
        avg_grad_norm = np.mean(grad_norms) if grad_norms else 0

        return avg_loss, avg_grad_norm

    def validate(self):
        """验证模型"""
        self.model.eval()
        total_loss = 0
        total_samples = 0

        with torch.no_grad():
            for sequences, targets in tqdm(self.valid_loader, desc='验证中', leave=False):
                sequences, targets = sequences.to(self.device), targets.to(self.device)

                outputs, _ = self.model(sequences)
                loss = self.criterion(outputs[:, -1, :], targets)

                total_loss += loss.item() * sequences.size(0)
                total_samples += sequences.size(0)

        return total_loss / total_samples

    def should_stop_early(self, valid_loss):
        """检查是否应该早停"""
        if valid_loss < self.best_loss - self.min_delta:
            self.best_loss = valid_loss
            self.best_model_state = self.model.state_dict().copy()
            self.patience_counter = 0
            return False
        else:
            self.patience_counter += 1
            return self.patience_counter >= self.patience

    def train(self, num_epochs):
        """完整训练流程"""
        print('开始高级训练流程...')
        print(f'优化器: {type(self.optimizer).__name__}')
        print(f'学习率调度器: {type(self.scheduler).__name__}')
        print(f'梯度累积步数: {self.accumulation_steps}')
        print('='*60)

        for epoch in range(num_epochs):
            # 训练
            train_loss, grad_norm = self.train_epoch(epoch)

            # 验证
            valid_loss = self.validate()

            # 记录历史
            self.history['train_losses'].append(train_loss)
            self.history['valid_losses'].append(valid_loss)
            self.history['learning_rates'].append(self.optimizer.param_groups[0]['lr'])
            self.history['grad_norms'].append(grad_norm)

            # 打印结果
            print(f'\nEpoch {epoch+1}:')
            print(f'  训练损失: {train_loss:.4f} | 训练困惑度: {math.exp(train_loss):.2f}')
            print(f'  验证损失: {valid_loss:.4f} | 验证困惑度: {math.exp(valid_loss):.2f}')
            print(f'  学习率: {self.optimizer.param_groups[0]["lr"]:.6f}')
            print(f'  梯度范数: {grad_norm:.4f}')

            # 早停检查
            if self.should_stop_early(valid_loss):
                print(f'\n早停触发! 验证损失连续{self.patience}轮未改善')
                break

            if valid_loss == self.best_loss:
                print('  新的最佳模型!')

            print('-'*60)

        # 加载最佳模型
        if self.best_model_state is not None:
            self.model.load_state_dict(self.best_model_state)
            print('\n已加载最佳模型权重')

        return self.history

print('高级训练器定义完成！')
print('\n新增特性:')
print('• AdamW优化器（权重衰减）')
print('• 余弦退火学习率调度')
print('• 梯度累积')
print('• 改进的早停机制')
print('• 梯度范数监控')

In [ ]:
# 演示高级训练（可选运行，因为之前已经训练过模型）
run_advanced_training = False  # 设置为True来运行高级训练

if run_advanced_training:
    print('开始演示高级训练技术...')

    # 创建高级模型（使用较小的规模进行演示）
    advanced_model = AdvancedLSTMLanguageModel(
        vocab_size=processor.vocab_size,
        embedding_dim=EMBEDDING_DIM,
        hidden_dim=EMBEDDING_DIM,  # 使用权重绑定需要相同维度
        num_layers=2,
        dropout_rate=0.3,
        tie_weights=True,
        use_layer_norm=True
    ).to(device)

    print(f'高级模型参数数量: {count_parameters(advanced_model):,}')

    # 创建高级训练器
    trainer = AdvancedTrainer(advanced_model, train_loader, valid_loader, device)

    # 开始训练
    advanced_history = trainer.train(num_epochs=5)  # 较少轮数用于演示

    # 可视化高级训练结果
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))

    epochs = range(1, len(advanced_history['train_losses']) + 1)

    # 损失曲线
    axes[0, 0].plot(epochs, advanced_history['train_losses'], 'b-', label='训练损失')
    axes[0, 0].plot(epochs, advanced_history['valid_losses'], 'r-', label='验证损失')
    axes[0, 0].set_title('高级训练损失曲线')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)

    # 学习率变化
    axes[0, 1].plot(epochs, advanced_history['learning_rates'], 'g-')
    axes[0, 1].set_title('余弦退火学习率调度')
    axes[0, 1].set_yscale('log')
    axes[0, 1].grid(True, alpha=0.3)

    # 梯度范数
    axes[1, 0].plot(epochs, advanced_history['grad_norms'], 'purple')
    axes[1, 0].set_title('梯度范数监控')
    axes[1, 0].grid(True, alpha=0.3)

    # 困惑度对比
    train_ppl = [math.exp(loss) for loss in advanced_history['train_losses']]
    valid_ppl = [math.exp(loss) for loss in advanced_history['valid_losses']]
    axes[1, 1].plot(epochs, train_ppl, 'b-', label='训练困惑度')
    axes[1, 1].plot(epochs, valid_ppl, 'r-', label='验证困惑度')
    axes[1, 1].set_title('困惑度变化')
    axes[1, 1].legend()
    axes[1, 1].set_yscale('log')
    axes[1, 1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

else:
    print('高级训练演示跳过（设置run_advanced_training=True来运行）')
    print('\n高级训练技术总结:')
    print('• 权重绑定可以减少参数数量并提高泛化能力')
    print('• 层归一化有助于训练稳定性')
    print('• 余弦退火学习率调度可以帮助模型跳出局部最优')
    print('• 梯度累积允许使用更大的有效批次大小')
    print('• AdamW优化器的权重衰减有助于正则化')
    print('• 梯度范数监控帮助诊断训练问题')

## 11. 模型分析和解释

分析训练好的模型，理解其学到的特征和模式。

In [ ]:
# 分析模型的词嵌入
def analyze_embeddings(model, processor, num_words=20):
    """分析词嵌入的相似性"""
    model.eval()

    # 获取嵌入权重
    embeddings = model.embedding.weight.data.cpu().numpy()

    # 计算词汇之间的余弦相似度
    from sklearn.metrics.pairwise import cosine_similarity
    similarity_matrix = cosine_similarity(embeddings)

    print('=== 词嵌入相似性分析 ===')
    print()

    # 选择一些常见词进行分析
    common_words = ['the', 'and', 'of', 'in', 'to', 'a', 'is', 'was', 'for', 'on']

    for word in common_words:
        if word in processor.word_to_idx:
            word_idx = processor.word_to_idx[word]
            similarities = similarity_matrix[word_idx]

            # 找到最相似的词（排除自己）
            similar_indices = np.argsort(similarities)[::-1][1:6]  # 前5个最相似的词

            print(f'与 "{word}" 最相似的词:')
            for i, idx in enumerate(similar_indices, 1):
                similar_word = processor.idx_to_word[idx]
                similarity_score = similarities[idx]
                print(f'  {i}. {similar_word} (相似度: {similarity_score:.3f})')
            print()

analyze_embeddings(model, processor)

# 分析模型的注意力模式（通过LSTM隐藏状态）
def analyze_hidden_states(model, processor, text, device):
    """分析LSTM隐藏状态的变化"""
    model.eval()

    tokens = processor.encode(text)
    if len(tokens) > SEQUENCE_LENGTH:
        tokens = tokens[:SEQUENCE_LENGTH]
    elif len(tokens) < SEQUENCE_LENGTH:
        pad_idx = processor.word_to_idx[processor.pad_token]
        tokens = [pad_idx] * (SEQUENCE_LENGTH - len(tokens)) + tokens

    input_tensor = torch.tensor([tokens], dtype=torch.long).to(device)

    with torch.no_grad():
        # 获取每一步的隐藏状态
        embedded = model.embedding(input_tensor)
        hidden = None
        hidden_states = []

        for i in range(embedded.size(1)):
            lstm_input = embedded[:, i:i+1, :]
            lstm_out, hidden = model.lstm(lstm_input, hidden)
            hidden_states.append(hidden[0].cpu().numpy())  # 只取h，不取c

    # 可视化隐藏状态的变化
    hidden_states = np.concatenate(hidden_states, axis=1)  # (num_layers, seq_len, hidden_dim)

    # 只显示第一层的隐藏状态
    first_layer_hidden = hidden_states[0, :, :50]  # 只显示前50个维度

    plt.figure(figsize=(12, 6))
    plt.imshow(first_layer_hidden.T, cmap='RdBu', aspect='auto')
    plt.colorbar()
    plt.title('LSTM隐藏状态激活模式')
    plt.xlabel('时间步')
    plt.ylabel('隐藏单元维度')

    # 添加单词标签
    words = [processor.idx_to_word[idx] for idx in tokens[-20:]]  # 显示最后20个词
    plt.xticks(range(len(tokens)-20, len(tokens)), words, rotation=45)

    plt.tight_layout()
    plt.show()

    return hidden_states

# 分析一个示例文本
sample_text = "The ancient civilization of Egypt was known for its magnificent pyramids and advanced knowledge"
print(f'\n分析文本: "{sample_text}"')
hidden_analysis = analyze_hidden_states(model, processor, sample_text, device)

## 12. 总结和下一步

总结本教程的主要内容，并提供进一步学习的方向。

## 教程总结

### 我们完成了什么

在这个综合性的LSTM文本预测教程中，我们系统地学习了：

#### 1. **理论基础**
- LSTM的工作原理和门控机制
- 语言建模的基本概念
- 文本预处理和编码技术

#### 2. **数据处理**
- WikiText-2数据集的下载和预处理
- 词汇表构建和文本编码
- 数据集的统计分析和可视化
- 序列数据的批量处理

#### 3. **模型实现**
- 基础LSTM语言模型的完整实现
- 高级LSTM模型（包含权重绑定、层归一化等技术）
- 灵活的模型架构设计

#### 4. **训练策略**
- 基础训练循环的实现
- 高级优化技术（AdamW、余弦退火、梯度累积）
- 早停和模型保存机制
- 训练过程的可视化

#### 5. **模型评估**
- 困惑度计算和解释
- 文本生成和质量评估
- 模型内部分析（嵌入、隐藏状态）

### 最终模型性能

我们的模型在WikiText-2数据集上取得了不错的性能，展示了LSTM在语言建模任务中的有效性。

---

## 下一步学习方向

### 1. **模型架构改进**
- Transformer架构（自注意力机制）
- GRU vs LSTM 对比实验
- 双向LSTM
- 多头注意力机制
- 残差连接

### 2. **高级训练技术**
- 混合精度训练（FP16）
- 分布式训练
- 知识蒸馏
- 对抗训练
- 课程学习

### 3. **更大规模实验**
- 更大的数据集（WikiText-103, BookCorpus）
- 多语言模型训练
- 预训练 + 微调范式
- 零样本和少样本学习

### 4. **应用扩展**
- 机器翻译
- 文本摘要
- 对话系统
- 代码生成
- 创意写作辅助

### 5. **评估和分析**
- BLEU、ROUGE等自动评估指标
- 人工评估实验设计
- 偏见和公平性分析
- 可解释性研究

---

## 实践建议

### 对于初学者：
1. **扎实基础**: 确保理解LSTM的每个组件
2. **动手实践**: 修改超参数，观察性能变化
3. **小数据集**: 在小数据集上快速实验
4. **可视化**: 多使用图表理解模型行为

### 对于进阶学习者：
1. **阅读论文**: 跟进最新的语言模型研究
2. **开源贡献**: 参与相关开源项目
3. **竞赛参与**: 参加NLP相关的机器学习竞赛
4. **产业应用**: 将模型部署到实际应用中

---

## 推荐资源

### 经典论文：
- "Long Short-Term Memory" (Hochreiter & Schmidhuber, 1997)
- "Attention Is All You Need" (Vaswani et al., 2017)
- "BERT: Pre-training of Deep Bidirectional Transformers" (Devlin et al., 2018)
- "Language Models are Unsupervised Multitask Learners" (GPT-2 paper)

### 实用工具：
- **Hugging Face Transformers**: 预训练模型库
- **Weights & Biases**: 实验跟踪和可视化
- **TensorBoard**: 训练监控
- **spaCy**: 高级文本处理

### 学习平台：
- **CS224N**: 斯坦福大学的NLP课程
- **fast.ai**: 实用的深度学习课程
- **Coursera**: 深度学习专项课程
- **Papers With Code**: 论文和代码实现

---

## 结语

恭喜您完成了这个全面的LSTM文本预测教程！您现在具备了：

- **扎实的理论基础** - 理解LSTM和语言建模的核心概念  
- **实践技能** - 能够独立实现和训练LSTM模型  
- **优化经验** - 掌握多种高级训练技巧  
- **分析能力** - 能够评估和改进模型性能  

自然语言处理是一个快速发展的领域，保持持续学习的心态，关注最新的研究进展，多动手实践，您将在这个激动人心的领域取得更大的成就！

**记住**: 最好的学习方法是实践。尝试修改代码，实验不同的想法，构建自己的项目。每一次失败都是学习的机会，每一个成功都是前进的动力。

祝您在NLP和深度学习的学习之路上一帆风顺！

---
*本教程由Claude Code生成，旨在提供全面的LSTM文本建模学习体验。*

## 📋 数据集更新说明

### 🔄 主要更新内容

本notebook已经成功更新为使用**本地WikiText token文件**，主要改进如下：

#### 1. **数据源更改**
- **原来**: 从网络下载WikiText-2数据集
- **现在**: 使用本地token文件
  - 训练数据: `/Users/xiaotingzhou/Documents/Lectures/ML_DL/LSTM/wiki.train.tokens`
  - 验证数据: `/Users/xiaotingzhou/Documents/Lectures/ML_DL/LSTM/wiki.valid.tokens`
  - 测试数据: `/Users/xiaotingzhou/Documents/Lectures/ML_DL/LSTM/wiki.test.tokens`

#### 2. **性能优化**
- **序列长度**: 从50减少到30，降低内存需求
- **模型规模**: 减小embedding和hidden维度，提高训练稳定性
- **数据限制**: 限制最大序列数量为50,000，确保合理训练时间
- **批次优化**: 设置num_workers=0，避免多进程问题

#### 3. **鲁棒性增强**
- **全面错误处理**: 每个关键步骤都添加了try-catch块
- **数值稳定性**: 检查和处理无效损失值和梯度
- **训练监控**: 实时监控训练进度和成功率
- **自动保存**: 自动保存最佳模型和中断模型
- **内存管理**: 自动清理GPU内存

#### 4. **用户体验改进**
- **详细日志**: 提供详细的训练进度信息
- **文件检查**: 启动前验证所有数据文件存在
- **灵活配置**: 可以轻松切换完整数据集或小数据集模式
- **进度可视化**: 使用tqdm显示训练和评估进度

### 🛠️ 配置参数

| 参数 | 原值 | 新值 | 说明 |
|------|------|------|------|
| 序列长度 | 50 | 30 | 减少内存使用 |
| 嵌入维度 | 256 | 128 | 降低模型复杂度 |
| 隐藏维度 | 512 | 256 | 提高训练稳定性 |
| 训练轮数 | 20 | 15 | 适中的训练时间 |
| 学习率 | 0.001 | 0.002 | 稍微提高收敛速度 |
| 最大序列数 | 无限制 | 50,000 | 控制训练时间 |

### 🚀 执行建议

1. **首次运行**: 确保所有依赖已安装，特别是PyTorch和torchtext
2. **内存不足**: 如遇内存问题，可将`USE_FULL_DATASET`设为`False`
3. **训练中断**: 可以从保存的检查点恢复训练
4. **GPU加速**: 自动检测并使用可用的GPU
5. **调试模式**: 可以通过减少`NUM_EPOCHS`进行快速测试

### ✅ 验证步骤

更新后的notebook包含多层验证：
- 数据文件存在性检查
- 数据质量验证
- 模型结构测试
- 前向传播验证
- 训练过程监控

这些改进确保了notebook能够稳定运行并产生有意义的训练结果。